In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

from src.data.load_data import DATA_PROCESSED_PATH
from src.data.splits import split_train_test, get_cv_splitter, validate_split_balance
from src.features.preprocessor import build_full_pipeline, get_preprocessing_columns

In [3]:
# Carregar dataset com features derivadas (gerado na Etapa 5)
df = pd.read_parquet(DATA_PROCESSED_PATH / "creditcard_features.parquet")
print(f" Shape: {df.shape}")
print(f" Distribuição: {df['Class'].value_counts().to_dict()}")

 Shape: (284807, 36)
 Distribuição: {0: 284315, 1: 492}


In [5]:
# Split treino/teste com estratificação
X_train, X_test, y_train, y_test = split_train_test(df)

print(f" Tamanhos:")
print(f" X_train: {X_train.shape}")
print(f" X_test:  {X_test.shape}")
print(f" Total:   {len(X_train) + len(X_test):,}")

# Validar balanceamento
balance = validate_split_balance(y_train, y_test)
print(f"\n Validação de estratificação:")
for k, v in balance.items():
    print(f"  {k}: {v}")

 Tamanhos:
 X_train: (227845, 35)
 X_test:  (56962, 35)
 Total:   284,807

 Validação de estratificação:
  train_fraud_pct: 0.1729
  test_fraud_pct: 0.172
  diff_pct_points: 0.0009
  within_tolerance: True


In [7]:
# Construir pipeline com Logistic Regression como teste
pipeline = build_full_pipeline(
    LogisticRegression(max_iter=1000, random_state=42)
)

print("Estrutura do pipeline:")
print(pipeline)

Estrutura do pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('scale', StandardScaler(),
                                                  ['Amount_log']),
                                                 ('pass', 'passthrough',
                                                  ['V1', 'V2', 'V3', 'V4', 'V5',
                                                   'V6', 'V7', 'V8', 'V9',
                                                   'V10', 'V11', 'V12', 'V13',
                                                   'V14', 'V15', 'V16', 'V17',
                                                   'V18', 'V19', 'V20', 'V21',
                                                   'V22', 'V23', 'V24', 'V25',
                                                   'V26', 'V27', 'V28',
                                                   'Hour_sin', 'Hour_cos', ...])],
                                   verbose_feature_names_out=False)),
                ('classifier',

In [8]:
# Pegar configuração de colunas
cols = get_preprocessing_columns()
print(f"\n Colunas que serão padronizadas: {cols['to_scale']}")
print(f"\n Colunas passthrough: {cols['passthrough'][:5]} ... (total: {len(cols['passthrough'])})")
print(f"\n Colunas descartadas: Amount, Time, Hour")


 Colunas que serão padronizadas: ['Amount_log']

 Colunas passthrough: ['V1', 'V2', 'V3', 'V4', 'V5'] ... (total: 31)

 Colunas descartadas: Amount, Time, Hour


In [9]:
# Vamos comparar duas abordagens para mostrar a diferença
from sklearn.preprocessing import StandardScaler

# ABORDAGEM ERRADA (com leakage)
print("ABORDAGEM ERRADA — fit em todo o dataset:")
scaler_wrong = StandardScaler()
all_amounts = df[['Amount_log']].values
scaler_wrong.fit(all_amounts)
print(f" Mean aprendido: {scaler_wrong.mean_[0]:.6f}")
print(f" Std aprendido:  {scaler_wrong.scale_[0]:.6f}")
print(f" Esses números incluem informação do TESTE!\n")

ABORDAGEM ERRADA — fit em todo o dataset:
 Mean aprendido: 3.152188
 Std aprendido:  1.656645
 Esses números incluem informação do TESTE!



In [10]:
# ABORDAGEM CORRETA (sem leakage) 
print(" ABORDAGEM CORRETA — fit APENAS no treino:")
scaler_right = StandardScaler()
scaler_right.fit(X_train[['Amount_log']])
print(f" Mean aprendido: {scaler_right.mean_[0]:.6f}")
print(f" Std aprendido:  {scaler_right.scale_[0]:.6f}")
print(f" Apenas dados de TREINO foram usados.\n")

 ABORDAGEM CORRETA — fit APENAS no treino:
 Mean aprendido: 3.153058
 Std aprendido:  1.655992
 Apenas dados de TREINO foram usados.



In [11]:
# Diferença
diff_mean = abs(scaler_wrong.mean_[0] - scaler_right.mean_[0])
diff_std = abs(scaler_wrong.scale_[0] - scaler_right.scale_[0])
print(f" Diferença numérica:")
print(f" |mean|: {diff_mean:.6f}")
print(f" |std|:  {diff_std:.6f}")
print(f"\n Em datasets grandes a diferença é pequena, mas em produção pode ser fatal.")
print(f" O importante é que o Pipeline GARANTE que isso nunca aconteça.")

 Diferença numérica:
 |mean|: 0.000870
 |std|:  0.000654

 Em datasets grandes a diferença é pequena, mas em produção pode ser fatal.
 O importante é que o Pipeline GARANTE que isso nunca aconteça.


In [12]:
# Splitter de validação cruzada
cv = get_cv_splitter(n_splits=5)

print(" Demonstração: distribuição de fraudes em cada fold")
print("=" * 60)
for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    fold_train_fraud = y_train.iloc[train_idx].mean() * 100
    fold_val_fraud = y_train.iloc[val_idx].mean() * 100
    fold_val_size = len(val_idx)
    print(f"  Fold {fold_idx+1}: "
          f"train={fold_train_fraud:.4f}% fraude | "
          f"val={fold_val_fraud:.4f}% fraude (n={fold_val_size:,})")

print("\n Estratificação preservada em todos os folds.")

 Demonstração: distribuição de fraudes em cada fold
  Fold 1: train=0.1734% fraude | val=0.1712% fraude (n=45,569)
  Fold 2: train=0.1728% fraude | val=0.1734% fraude (n=45,569)
  Fold 3: train=0.1728% fraude | val=0.1734% fraude (n=45,569)
  Fold 4: train=0.1728% fraude | val=0.1734% fraude (n=45,569)
  Fold 5: train=0.1728% fraude | val=0.1734% fraude (n=45,569)

 Estratificação preservada em todos os folds.


In [13]:
# Salvar arrays para reuso na Etapa 7 (modelagem)
import joblib

splits_dir = DATA_PROCESSED_PATH / "splits"
splits_dir.mkdir(exist_ok=True)

joblib.dump(X_train, splits_dir / "X_train.pkl")
joblib.dump(X_test, splits_dir / "X_test.pkl")
joblib.dump(y_train, splits_dir / "y_train.pkl")
joblib.dump(y_test, splits_dir / "y_test.pkl")

print("Splits salvos em data/processed/splits/:")
for f in splits_dir.iterdir():
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   {f.name} ({size_mb:.2f} MB)")

Splits salvos em data/processed/splits/:
   X_test.pkl (15.43 MB)
   X_train.pkl (61.71 MB)
   y_test.pkl (1.74 MB)
   y_train.pkl (6.95 MB)
